# 🎨 W Collection - Ürün Açıklaması Prototip

**Amaç:** Florence-2 ile görsel analizi → LLM ile Türkçe açıklama

**Örnek:** "Yüzde yüz pamuk kumaştan üretilen kahve renkli W Collection kazak; düğmeli polo yakaya sahiptir."

## 1️⃣ Kurulum (İlk Sefer Bir Kere)

In [ ]:
!pip install -q transformers pillow google-generativeai einops timm

## 2️⃣ Gemini API Key (Ücretsiz!)

1. Git: https://makersuite.google.com/app/apikey
2. API key al
3. Colab Secrets'a ekle: Sol menü → 🔑 Secrets → `GEMINI_API_KEY`

In [ ]:
from google.colab import userdata
import google.generativeai as genai

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)

print("✅ Gemini hazır")

## 3️⃣ Florence-2 Yükle

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoProcessor
from PIL import Image
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if device == "cuda" else torch.float32

print(f"Device: {device}")
print("Florence-2 yükleniyor...")

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Florence-2-base",
    torch_dtype=torch_dtype,
    trust_remote_code=True,
).to(device)

processor = AutoProcessor.from_pretrained(
    "microsoft/Florence-2-base",
    trust_remote_code=True,
)

print("✅ Florence-2 hazır")

## 4️⃣ Ana Fonksiyonlar

In [ ]:
def florence_analyze(image, task="<MORE_DETAILED_CAPTION>"):
    """Florence-2 ile görsel analizi"""
    inputs = processor(text=task, images=image, return_tensors="pt").to(device, torch_dtype)
    
    generated_ids = model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=1024,
        num_beams=3,
    )
    
    result = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    return result.replace("</s>", "").replace(task, "").strip()


def analyze_product(image_path):
    """Ürün görselini analiz et - 4 farklı analiz"""
    image = Image.open(image_path).convert("RGB")
    
    print("🔍 Analiz ediliyor...")
    
    results = {
        'caption': florence_analyze(image, "<MORE_DETAILED_CAPTION>"),
        'objects': florence_analyze(image, "<OD>"),
        'regions': florence_analyze(image, "<DENSE_REGION_CAPTION>"),
        'ocr': florence_analyze(image, "<OCR>")
    }
    
    print("✅ Analiz tamam")
    return results, image


def generate_description(analysis, product_info=None):
    """LLM ile Türkçe açıklama üret"""
    
    prompt = f"""Sen W Collection için ürün açıklamaları yazıyorsun.

## Görsel Analiz:
- Detaylı açıklama: {analysis['caption']}
- Nesneler: {analysis['objects']}
- Bölgesel detaylar: {analysis['regions']}
- Yazılar: {analysis['ocr']}
"""

    if product_info:
        prompt += "\n## Ek Bilgiler:\n"
        for key, value in product_info.items():
            prompt += f"- {key}: {value}\n"
    
    prompt += """\n## Görev:
Yukarıdaki verilere göre W Collection için doğal, akıcı Türkçe ürün açıklaması yaz.

**Format:** 2-3 cümle, düz metin (markdown yok)

**Örnek:** "Yüzde yüz pamuk kumaştan üretilen kahve renkli W Collection kazak; düğmeli polo yakaya sahiptir. Günlük kullanım için ideal bir seçenektir."

**Kurallar:**
- Doğal ve akıcı Türkçe
- Görsel analizden çıkardığın detayları kullan
- Abartma, sadece gördüklerine dayanarak yaz
- Markdown, başlık kullanma

Şimdi açıklamayı yaz:
"""
    
    print("✍️  Açıklama oluşturuluyor...")
    
    model = genai.GenerativeModel('gemini-1.5-flash')
    response = model.generate_content(prompt)
    
    print("✅ Açıklama hazır")
    return response.text.strip()


print("✅ Fonksiyonlar hazır")

## 5️⃣ Test Et - Görsel Yükle

In [ ]:
from google.colab import files

# Görseli yükle
print("📤 Lütfen ürün görselini yükleyin:")
uploaded = files.upload()
image_path = list(uploaded.keys())[0]

print(f"✅ Görsel yüklendi: {image_path}")

## 6️⃣ Analiz Et

In [ ]:
# Görseli analiz et
analysis, image = analyze_product(image_path)

# Görseli göster
plt.figure(figsize=(8, 8))
plt.imshow(image)
plt.axis('off')
plt.title('Ürün Görseli')
plt.show()

# Analiz sonuçları
print("\n" + "="*80)
print("FLORENCE-2 ANALİZ SONUÇLARI")
print("="*80)
print(f"\n📝 Caption:\n{analysis['caption']}")
print(f"\n🎯 Objects:\n{analysis['objects']}")
print(f"\n🔍 Regions:\n{analysis['regions']}")
print(f"\n📄 OCR:\n{analysis['ocr']}")

## 7️⃣ Açıklama Üret

In [ ]:
# Ürün bilgilerini gir (opsiyonel)
product_info = {
    "Kategori": "kazak",
    "Malzeme": "%100 pamuk",
    "Renk": "kahverengi",
    # İstersen daha fazla ekle:
    # "Beden": "M",
    # "Koleksiyon": "2024 Sonbahar"
}

# Açıklama üret
description = generate_description(analysis, product_info)

# Sonuç
print("\n" + "="*80)
print("✨ ÜRÜN AÇIKLAMASI")
print("="*80)
print(f"\n{description}\n")

---

## 🚀 Hızlı Test Fonksiyonu (Hepsini Tek Seferde)

In [ ]:
def quick_test(image_path, **product_info):
    """Tek satırda her şeyi yap"""
    # Analiz
    analysis, image = analyze_product(image_path)
    
    # Görsel
    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.axis('off')
    plt.show()
    
    # Açıklama
    description = generate_description(analysis, product_info if product_info else None)
    
    print("\n" + "="*80)
    print("✨ SONUÇ")
    print("="*80)
    print(f"\n{description}\n")
    
    return description

# Kullanım:
# desc = quick_test(image_path, Kategori="kazak", Malzeme="%100 pamuk", Renk="kahverengi")

---

## 🔄 Toplu İşlem (Çoklu Ürünler)

In [ ]:
# Birden fazla görsel yükle
print("📤 Birden fazla görsel yükleyin (Ctrl tuşuna basılı tutarak):")
uploaded_files = files.upload()

results = []

for filename in uploaded_files.keys():
    print(f"\n{'='*80}")
    print(f"İşleniyor: {filename}")
    print(f"{'='*80}")
    
    try:
        analysis, image = analyze_product(filename)
        description = generate_description(analysis)
        
        results.append({
            'filename': filename,
            'description': description,
            'status': 'success'
        })
        
        print(f"\n✅ {filename}:")
        print(description)
        
    except Exception as e:
        print(f"\n❌ {filename}: Hata - {str(e)}")
        results.append({
            'filename': filename,
            'error': str(e),
            'status': 'failed'
        })

# Özet
print(f"\n{'='*80}")
print("ÖZET")
print(f"{'='*80}")
success = sum(1 for r in results if r['status'] == 'success')
print(f"Toplam: {len(results)} | Başarılı: {success} | Hatalı: {len(results) - success}")

---

## 💾 Sonuçları Kaydet ve İndir

In [ ]:
import json

# JSON olarak kaydet
with open('product_descriptions.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

# İndir
files.download('product_descriptions.json')

print("✅ Sonuçlar kaydedildi ve indiriliyor...")

---

## 🎯 Prompt'u Özelleştir (Deney Yap)

In [ ]:
def custom_prompt_test(analysis, style="normal"):
    """Farklı stil dene"""
    
    styles = {
        "normal": "2-3 cümle, doğal ve akıcı",
        "kisa": "Sadece 1 cümle, kısa ve öz",
        "detayli": "4-5 cümle, çok detaylı ve satış odaklı",
        "teknik": "Sadece teknik özellikleri listele"
    }
    
    prompt = f"""Florence-2 analizi:
- Caption: {analysis['caption']}
- Objects: {analysis['objects']}

W Collection için Türkçe ürün açıklaması yaz.
Stil: {styles[style]}

Açıklama:
"""
    
    model = genai.GenerativeModel('gemini-1.5-flash')
    response = model.generate_content(prompt)
    
    return response.text.strip()

# Test et
print("NORMAL:")
print(custom_prompt_test(analysis, "normal"))
print("\nKISA:")
print(custom_prompt_test(analysis, "kisa"))
print("\nDETAYLI:")
print(custom_prompt_test(analysis, "detayli"))

---

## 📝 Notlar

**Prototip için şu an yeterli. İlerisi için:**
- [ ] Farklı LLM'ler test et (Claude, GPT)
- [ ] Florence-2-large dene (daha detaylı)
- [ ] Prompt mühendisliği ile çıktıyı optimize et
- [ ] Batch processing'i optimize et
- [ ] Google Drive entegrasyonu
- [ ] CSV export

**Hızlı Kullanım:**
```python
# 1. Görsel yükle
uploaded = files.upload()
image_path = list(uploaded.keys())[0]

# 2. Test et
quick_test(image_path, Kategori="kazak", Malzeme="%100 pamuk")
```